In [ ]:
# 测试 unsloth
import unsloth
print(unsloth.__version__)

In [ ]:
# 加载并量化基础模型
from unsloth import FastLanguageModel
from transformers import BitsAndBytesConfig

random_seed = 42
max_seq_length = 2048

# 量化方式
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# 加载量化模型和分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./base_model/Qwen3-8B",
    max_seq_length=max_seq_length,
    quantization_config=quantization_config,
)

In [ ]:
# 配置模型以进行PEFT(Parameter Efficient Fine-Tuning，参数高效微调)
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=16, # LoRA 的秩（rank）
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ], # 指定要应用 LoRA 的模块名称
    lora_alpha=16, # LoRA 中的缩放系数
    lora_dropout=0, # LoRA 部分的 dropout 概率
    bias="none", # 不微调 bias 参数
    use_gradient_checkpointing="unsloth", # 启用梯度检查点以节省显存
    random_state=random_seed, # LoRA 初始化的随机种子
    use_rslora=False, # 不启用 Rank Stabilized LoRA (RS-LoRA)
    loftq_config=None, # 不配置 LoFTQ
)

In [ ]:
# 加载训练和验证数据集
import datasets

from unsloth import is_bfloat16_supported

from unsloth.chat_templates import get_chat_template

# 加载数据集
train_dataset = datasets.load_dataset("json", data_files={"train": "train.jsonl"}, split="train")
eval_dataset = datasets.load_dataset("json", data_files={"eval": "val.jsonl"}, split="eval")

# 使用对话模板格式化数据
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
def formatting_func(example):
    return {
        "text": f"<|im_start|>user\n{example['prompt'].strip()}<|im_end|>\n"
                f"<|im_start|>assistant\n{example['completion'].strip()}<|im_end|>"
    }
train_dataset = train_dataset.map(formatting_func)
eval_dataset  = eval_dataset.map(formatting_func)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
# 配置训练器
# 配置参数
args = TrainingArguments(
    ###### 训练参数
    seed = random_seed,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    warmup_steps = 5,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    ###### 数据类型
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    ###### 验证参数
    eval_strategy = "steps",
    eval_steps = 4,
    per_device_eval_batch_size = 1,
    ###### 输出参数
    logging_steps = 1,
    output_dir = "outputs",
)

# 使用参数配置训练器
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    max_seq_length = max_seq_length,
    args = args,
)

In [ ]:
# 微调
finetune_metrics = trainer.train()

# 以16bit精度保存模型
# model.save_pretrained_merged("./finetuned_model/Qwen3-8B", tokenizer, save_method="merged_16bit")

from peft import PeftModel
# 保存 adapter
model.save_pretrained("./finetuned_model/Qwen3-8B-adapter-only")
tokenizer.save_pretrained("./finetuned_model/Qwen3-8B-adapter-only")

In [ ]:
# 推理，加载微调后的模型
from transformers import TextStreamer
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained("./base_model/Qwen3-8B")
model.load_adapter("./finetuned_model/Qwen3-8B-adapter-only")
FastLanguageModel.for_inference(model)  # 开启推理优化

In [ ]:
# 应用提示模版并分词
input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": train_dataset["prompt"][19]}],  # 使用 TRL 对话格式
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# 根据用户输入生成模型输出
_ = model.generate(
    input_ids=input_ids,
    streamer=TextStreamer(tokenizer), # 生成时使用流式输出
    max_new_tokens=32768,
    do_sample=False, # 禁用随机采样，生成确定性输出
)

In [ ]:
# vllm serve base_model/Qwen3-8B --served-model-name Qwen3-8B-sft --max-model-len 2048 --enable-lora --lora-modules alora=finetuned_model/Qwen3-8B-adapter-only